# Session Retrospective Overlays

Compare multiple recorded VEYN sessions side-by-side. Overlay intent timelines,
biometric traces, and outcome annotations.

## Setup
```bash
pip install pandas matplotlib seaborn requests
```

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

DB_PATH = '../veyn.db'
conn = sqlite3.connect(DB_PATH)
print('Connected to', DB_PATH)

In [ ]:
# List available sessions
sessions = pd.read_sql_query("""
SELECT id, label, started_at, ended_at
FROM sessions
ORDER BY started_at DESC
LIMIT 20
""", conn)
sessions

In [ ]:
# Select two sessions to compare
SESSION_A = sessions.iloc[0]['id'] if len(sessions) > 0 else 'none'
SESSION_B = sessions.iloc[1]['id'] if len(sessions) > 1 else 'none'
print(f'Comparing: {SESSION_A} vs {SESSION_B}')

In [ ]:
# Load events for both sessions
def load_session_events(session_id):
    return pd.read_sql_query("""
    SELECT ts_ms, device_id, metric, value, unit
    FROM events
    WHERE session_id = ?
    ORDER BY ts_ms
    """, conn, params=[session_id])

ev_a = load_session_events(SESSION_A)
ev_b = load_session_events(SESSION_B)

# Normalise timestamps to relative seconds from session start
for df in [ev_a, ev_b]:
    if len(df) > 0:
        df['t_sec'] = (df['ts_ms'] - df['ts_ms'].min()) / 1000.0

print(f'Session A: {len(ev_a)} events, Session B: {len(ev_b)} events')

In [ ]:
# Overlay HR traces for both sessions
hr_a = ev_a[ev_a['metric'] == 'heart_rate']
hr_b = ev_b[ev_b['metric'] == 'heart_rate']

fig, ax = plt.subplots(figsize=(14, 5))
if len(hr_a) > 0:
    ax.plot(hr_a['t_sec'], hr_a['value'], label=f'Session A', linewidth=1.5)
if len(hr_b) > 0:
    ax.plot(hr_b['t_sec'], hr_b['value'], label=f'Session B', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Heart Rate (bpm)')
ax.set_title('Session A/B — Heart Rate Overlay')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Load memory records with outcomes for both sessions
mem_query = """
SELECT session_id, topic, summary, intent_at_time, hrv_at_time,
       outcome_rating, outcome_notes, confidence_at_time
FROM memory_records
WHERE session_id IN (?, ?)
ORDER BY timestamp_ms
"""
memories = pd.read_sql_query(mem_query, conn, params=[SESSION_A, SESSION_B])
memories

In [ ]:
# Outcome distribution per session
if 'outcome_rating' in memories.columns and memories['outcome_rating'].notna().any():
    fig, ax = plt.subplots(figsize=(8, 4))
    ct = memories.groupby(['session_id', 'outcome_rating']).size().unstack(fill_value=0)
    ct.plot.bar(ax=ax)
    ax.set_title('Outcome Distribution by Session')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No outcome ratings recorded yet.')

In [ ]:
conn.close()
print('Done.')